# 03. Unsupervised Segmentation

This notebook keeps the unsupervised section from the original work but packages it into a cleaner, repeatable analysis. It clusters apps in a reduced feature space and then flags outliers with Isolation Forest.


In [ ]:
from pathlib import Path

HELPER_NOTEBOOK = Path("00_helpers.ipynb")
if not HELPER_NOTEBOOK.exists():
    HELPER_NOTEBOOK = Path("notebooks/00_helpers.ipynb")

HELPER_NOTEBOOK

In [ ]:
%run $HELPER_NOTEBOOK


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [ ]:
data = load_or_prepare_data()
model_df = data.copy()


## Encode Modeling Features


In [ ]:
label_encoders = {}
for column in CATEGORICAL_COLUMNS:
    encoder = LabelEncoder()
    model_df[column] = encoder.fit_transform(model_df[column])
    label_encoders[column] = encoder


In [ ]:
X = model_df[MODEL_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pca_variance = pd.Series(pca.explained_variance_ratio_, index=["PC1", "PC2", "PC3"])
pca_variance


## K-Means Clustering


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
model_df["Cluster"] = kmeans.fit_predict(X_pca)
model_df["Cluster"].value_counts().sort_index()


In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=model_df["Cluster"], cmap="viridis", s=18, alpha=0.6)
ax.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    kmeans.cluster_centers_[:, 2],
    c="red",
    marker="X",
    s=220,
    label="Centroids",
)
ax.set_title("K-Means Clusters in PCA Space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.legend()
plt.show()


In [ ]:
cluster_profile = data.assign(Cluster=model_df["Cluster"]).groupby("Cluster")[
    ["Installs", "Price", "Size", "Rating", "Monetization Score"]
].median().round(2)
cluster_profile


## Outlier Detection


In [ ]:
isolation_features = [
    "Category",
    "Installs",
    "Free",
    "Price",
    "Size",
    "Content Rating",
    "Ad Supported",
    "In App Purchases",
    "Editors Choice",
    "Region",
    "Year",
    "Age",
    "Days Since Update",
    "Rating Confidence",
    "Season",
    "Monetization Score",
]

iso_forest = IsolationForest(contamination=0.10, random_state=42)
outlier_predictions = iso_forest.fit_predict(model_df[isolation_features])
model_df["Isolation Forest Outlier"] = outlier_predictions == -1
model_df["Isolation Forest Outlier"].value_counts()


In [ ]:
outliers = data.loc[model_df["Isolation Forest Outlier"]].copy()
outliers[[
    "App Name",
    "Category",
    "Installs",
    "Price",
    "Rating",
    "Monetization Score",
]].sort_values("Installs", ascending=False).head(10)
